In [ ]:
pip install pandas nltk transformers torch

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import wordnet
from transformers import pipeline
from tqdm import tqdm

# Download resource cần thiết
nltk.download('punkt')
nltk.download('wordnet')

# Load model BERT
unmasker = pipeline('fill-mask', model='bert-base-uncased')

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().replace('_', ' '))
    return list(synonyms)

def word_substitution(text, top_k=10):
    words = nltk.word_tokenize(text)
    freq_dist = nltk.FreqDist(words)
    top_words = [word.lower() for word, _ in freq_dist.most_common(top_k)]

    perturbed_words = words.copy()

    for i, word in enumerate(words):
        if word.lower() in top_words:
            synonyms = get_synonyms(word)
            if synonyms:
                perturbed_words[i] = synonyms[0]
            else:
                masked_words = words.copy()
                masked_words[i] = '[MASK]'
                masked_text = ' '.join(masked_words)
                result = unmasker(masked_text)
                if result:
                    perturbed_words[i] = result[0]['token_str']

    return ' '.join(perturbed_words)

# Đọc file dữ liệu
df = pd.read_csv('ielts_dataset_v1.4_rawxparaphrased.csv')

# Xử lý progress bar
tqdm.pandas()

# Áp dụng word_substitution có điều kiện
def conditional_perturb(row):
    if row['is_ai'] == 1 and row['variant'] == 'raw':
        return word_substitution(row['text'])
    else:
        return row['text']

# Apply với progress bar
df['perturbed_text'] = df.progress_apply(conditional_perturb, axis=1)

# Lưu file kết quả
df.to_csv('perturbed_essays.csv', index=False)
